Import Libraries

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

Check TensorFlow

In [3]:
print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.21.0


Dataset Path

In [4]:
dataset_path = r"D:\Infosys\Skincare dataset"

print(os.listdir(dataset_path))

['clear skin', 'dark spots', 'puffy eyes', 'wrinkles']


Count images

In [5]:
for folder in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, folder)
    print(folder, ":", len(os.listdir(folder_path)), "images")

clear skin : 300 images
dark spots : 303 images
puffy eyes : 300 images
wrinkles : 300 images


Load the images

In [6]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

In [7]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

In [8]:
train_generator = datagen.flow_from_directory(
    dataset_path,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

Found 963 images belonging to 4 classes.


In [9]:
validation_generator = datagen.flow_from_directory(
    dataset_path,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

Found 240 images belonging to 4 classes.


In [10]:
print(train_generator.class_indices)

{'clear skin': 0, 'dark spots': 1, 'puffy eyes': 2, 'wrinkles': 3}


Load MobileNetV2

In [16]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [17]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

In [18]:
base_model.trainable = False

In [19]:
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dense(4, activation="softmax")
])

In [20]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,468 (9.24 MB)

 Trainable params: 164,484 (642.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Compile the model

In [21]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

Train the model

In [22]:
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=10
)

Epoch 1/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 40s 1s/step - accuracy: 0.5057 - loss: 1.1608 - val_accuracy: 0.7125 - val_loss: 0.8126
Epoch 2/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 21s 691ms/step - accuracy: 0.7425 - loss: 0.7090 - val_accuracy: 0.7917 - val_loss: 0.5885
Epoch 3/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 22s 701ms/step - accuracy: 0.8131 - loss: 0.5510 - val_accuracy: 0.8292 - val_loss: 0.5095
Epoch 4/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 22s 695ms/step - accuracy: 0.8536 - loss: 0.4709 - val_accuracy: 0.8542 - val_loss: 0.4539
Epoch 5/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 22s 697ms/step - accuracy: 0.8744 - loss: 0.4111 - val_accuracy: 0.8542 - val_loss: 0.4341
Epoch 6/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 21s 692ms/step - accuracy: 0.8972 - loss: 0.3668 - val_accuracy: 0.8500 - val_loss: 0.4195
Epoch 7/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 22s 694ms/step - accuracy: 0.9034 - loss: 0.3351 - val_accuracy: 0.8583 - val_loss: 0.4039
Epoch 8/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 22s 712ms/step - accuracy: 0.9148 - loss: 0.3056 - val_accurac

Save the model

In [23]:
import os

save_path = r"D:\Infosys\AI-Skin-Intelligence-Personalized-Skincare-Planner\backend\ml\saved_models"

os.makedirs(save_path, exist_ok=True)

model.save(os.path.join(save_path, "skin_classifier.keras"))

print("Model saved successfully!")

Model saved successfully!
